# Crawl repo GitHub năm 2026 (≥ 10 sao, ưu tiên nhiều topic)

Notebook này thu thập 12.000 repo GitHub được tạo trong năm 2026, có ít nhất 10 sao,
ưu tiên repo có hơn 1 topic, rồi xuất ra file `github_repos_2026.parquet`.

**Trước khi chạy:** đặt biến môi trường `GITHUB_TOKEN` (xem ô cài đặt bên dưới).

## 1. Cài thư viện

In [1]:
%pip install -q requests pandas pyarrow tqdm

Note: you may need to restart the kernel to use updated packages.



[notice] A new release of pip is available: 26.2.1 -> 26.2.2
[notice] To update, run: python.exe -m pip install --upgrade pip


## 2. Cấu hình token
Đặt token trước khi chạy. Không đưa token thật vào notebook được chia sẻ.

In [2]:
import os
# Thay bằng token của bạn, hoặc đặt sẵn biến môi trường GITHUB_TOKEN ở máy
# os.environ["GITHUB_TOKEN"] = "ghp_xxxxxxxxxxxxxxxx"
print("Đã có token" if os.getenv("GITHUB_TOKEN") else "CHƯA có GITHUB_TOKEN")

Đã có token


## 3. Khai báo tham số và import

In [3]:
"""
Crawl repo GitHub được tạo trong năm 2026, có >= 10 sao,
ưu tiên repo có > 1 topic, lấy 12.000 dòng và xuất ra file .parquet.

Cách dùng:
    pip install requests pandas pyarrow tqdm
    export GITHUB_TOKEN=ghp_xxxxxxxx        # Windows CMD: set GITHUB_TOKEN=ghp_xxxxxxxx
    python crawl_github.py
"""

import datetime as dt
import os
import time

import pandas as pd
import requests
from tqdm import tqdm

API_URL = "https://api.github.com/search/repositories"
YEAR = 2026
MIN_STARS = 10
TARGET_ROWS = 12000
OVERFETCH = int(TARGET_ROWS * 1.3)   # thu dư ~30% để có đủ repo nhiều topic
WINDOW_DAYS = 7                      # chia nhỏ theo tuần để vượt giới hạn 1000 kết quả/truy vấn
STAR_BUCKETS = [(10, 19), (20, 49), (50, 99), (100, 249),
                (250, 499), (500, 999), (1000, 4999), (5000, 999999)]
SEARCH_SLEEP = 2.1                   # Search API giới hạn ~30 request/phút
OUT_FILE = "github_repos_2026.parquet"

## 4. Hàm kết nối và gọi API (có retry, tự chờ khi bị rate limit)

In [4]:
def make_session() -> requests.Session:
    token = os.getenv("GITHUB_TOKEN")
    if not token:
        raise SystemExit("Thiếu biến môi trường GITHUB_TOKEN")
    s = requests.Session()
    s.headers.update({
        "Accept": "application/vnd.github+json",
        "Authorization": f"Bearer {token}",
        "X-GitHub-Api-Version": "2022-11-28",
        "User-Agent": "github-repo-crawler",
    })
    return s

In [5]:
def get_json(session: requests.Session, params: dict, retries: int = 5):
    for attempt in range(retries):
        try:
            r = session.get(API_URL, params=params, timeout=30)
        except requests.RequestException as exc:
            print(f"Lỗi mạng: {exc}, thử lại...")
            time.sleep(2 ** attempt)
            continue

        if r.status_code == 200:
            return r.json()

        if r.status_code in (403, 429):  # hết hạn mức -> chờ
            if r.headers.get("Retry-After"):
                wait = int(r.headers["Retry-After"])
            elif r.headers.get("X-RateLimit-Reset"):
                wait = max(int(r.headers["X-RateLimit-Reset"]) - int(time.time()), 0) + 5
            else:
                wait = 60
            print(f"Bị giới hạn, chờ {wait} giây...")
            time.sleep(min(wait, 3600))
            continue

        if r.status_code == 422:  # thường là vượt 1000 kết quả
            return None

        print(f"HTTP {r.status_code}: {r.text[:200]}")
        return None
    return None

## 5. Chia nhỏ truy vấn theo thời gian và số sao

In [6]:
def date_windows():
    cur = dt.date(YEAR, 1, 1)
    last = dt.date(YEAR, 12, 31)
    while cur <= last:
        end = min(cur + dt.timedelta(days=WINDOW_DAYS - 1), last)
        yield cur, end
        cur = end + dt.timedelta(days=1)

In [7]:
def search_query(session: requests.Session, query: str):
    """Lấy tối đa 1000 kết quả (10 trang x 100) cho một truy vấn."""
    for page in range(1, 11):
        data = get_json(session, {
            "q": query,
            "sort": "stars",
            "order": "desc",
            "per_page": 100,
            "page": page,
        })
        time.sleep(SEARCH_SLEEP)
        if not data or not data.get("items"):
            return
        yield from data["items"]
        if len(data["items"]) < 100:
            return

## 6. Chuyển JSON của GitHub thành một dòng dữ liệu và chạy crawl

In [8]:
def to_row(item: dict, collected_at: str) -> dict:
    lic = item.get("license")
    return {
        "repo_id": item["id"],
        "owner": item["owner"]["login"],
        "repo_name": item["name"],
        "full_name": item["full_name"],
        "primary_language": item.get("language"),
        "description": item.get("description"),
        "topics": item.get("topics") or [],
        "license_key": lic.get("key") if lic else None,
        "is_fork": item.get("fork"),
        "is_archived": item.get("archived"),
        "created_at": item.get("created_at"),
        "updated_at": item.get("updated_at"),
        "pushed_at": item.get("pushed_at"),
        "collected_at": collected_at,
        "star_count": item.get("stargazers_count"),
        "forks_count": item.get("forks_count"),
        "subscribers_count": item.get("subscribers_count"),  # thường là None ở Search API
        "open_issues_count": item.get("open_issues_count"),
        "size_kb": item.get("size"),
        "has_issues": item.get("has_issues"),
        "has_wiki": item.get("has_wiki"),
        "default_branch": item.get("default_branch"),
    }

In [9]:
def crawl() -> list[dict]:
    session = make_session()
    collected_at = dt.datetime.now(dt.timezone.utc).strftime("%Y-%m-%dT%H:%M:%SZ")
    seen: set[int] = set()
    rows: list[dict] = []

    pbar = tqdm(total=OVERFETCH, desc="Crawl")
    for start, end in date_windows():
        for lo, hi in STAR_BUCKETS:
            if len(rows) >= OVERFETCH:
                break
            query = f"created:{start}..{end} stars:{lo}..{hi}"
            for item in search_query(session, query):
                if item["id"] in seen or item["stargazers_count"] < MIN_STARS:
                    continue
                seen.add(item["id"])
                rows.append(to_row(item, collected_at))
                pbar.update(1)
                if len(rows) >= OVERFETCH:
                    break
        if len(rows) >= OVERFETCH:
            break
    pbar.close()
    return rows

## 7. Làm sạch, ưu tiên repo có nhiều topic, chọn 12.000 dòng

In [10]:
def build_dataframe(rows: list[dict]) -> pd.DataFrame:
    df = pd.DataFrame(rows).drop_duplicates(subset="repo_id", keep="first")

    df["created_at"] = pd.to_datetime(df["created_at"], utc=True)
    for col in ["updated_at", "pushed_at", "collected_at"]:
        df[col] = pd.to_datetime(df[col], utc=True, errors="coerce")
    df = df[df["created_at"].dt.year == YEAR]

    df["topics"] = df["topics"].apply(lambda t: sorted({str(x).lower() for x in t if x}))
    df["topic_count"] = df["topics"].apply(len)
    df["is_multi_topic"] = df["topic_count"] > 1

    # Ưu tiên repo có > 1 topic, sau đó theo số sao giảm dần
    df = (df.sort_values(["is_multi_topic", "star_count"], ascending=[False, False])
            .head(TARGET_ROWS)
            .reset_index(drop=True))
    return df

## 8. Chạy crawl

In [11]:
raw_rows = crawl()
print(f'Thu thập được {len(raw_rows)} ứng viên')

Crawl: 100%|██████████| 15600/15600 [15:38<00:00, 16.62it/s]

Thu thập được 15600 ứng viên


## 9. Xuất file parquet

In [12]:
df = build_dataframe(raw_rows)
df.to_parquet(OUT_FILE, index=False, engine="pyarrow")
print(f"Đã lưu {len(df)} dòng x {df.shape[1]} cột vào {OUT_FILE}")
print(f"Repo có > 1 topic: {int(df['is_multi_topic'].sum())}")
if len(df) < TARGET_ROWS:
    print(f"Cảnh báo: chỉ đạt {len(df)}/{TARGET_ROWS} dòng")

Đã lưu 12000 dòng x 24 cột vào github_repos_2026.parquet
Repo có > 1 topic: 5506


## 10. Xem thử kết quả

In [14]:
import pandas as pd
df = pd.read_parquet(OUT_FILE)
print(df.shape)
df[['full_name','primary_language','star_count','topic_count','created_at']].head(10)

df.to_csv("github_repos_2026.csv", index=False, encoding="utf-8-sig")

(12000, 24)
